In [1]:
"""
AMBS heuristic with an interface parallel to the DeepRL script (SET1 / SET2 / SET3),
and evaluation under identical conditions:

- same per-run seeds as PPO_Hier.test (seed_base + r),
- same discrete-uniform demand model as CLSPHierEnv._setup_demand_params(),
- same initialization as CLSPHierEnv.reset(): I = 0 and a random initial setup iSU sampled with rng(seed).
"""

import os
import math
import numpy as np
from tqdm import tqdm

# ============================================================
# 0) Global configuration (mirrors DeepRL for reproducibility)
# ============================================================

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
np.random.seed(SEED)

# ============================================================
# 1) Experiment set selection (aligned with DeepRL conventions)
# ============================================================
# Choose which paper instance to run (S-CLSP only):
# Examples: "1.1", "1.15", "1.35", "2.3", "2.12"
INSTANCE_ID = "2.8"


# Paper instance registry (ID -> builder parameters)

# Case I: homogeneous demand, K in {7..15}, cf in {1.1, 1.5}, cov in {low, high}
CASE_I_REGISTRY = {
    # -------------------------
    # Low demand variability
    # -------------------------
    "1.1":  dict(K=7,  cf=1.1, cov="low"),
    "1.2":  dict(K=7,  cf=1.5, cov="low"),
    "1.3":  dict(K=8,  cf=1.1, cov="low"),
    "1.4":  dict(K=8,  cf=1.5, cov="low"),
    "1.5":  dict(K=9,  cf=1.1, cov="low"),
    "1.6":  dict(K=9,  cf=1.5, cov="low"),
    "1.7":  dict(K=10, cf=1.1, cov="low"),
    "1.8":  dict(K=10, cf=1.5, cov="low"),

    "1.17": dict(K=11, cf=1.1, cov="low"),
    "1.18": dict(K=11, cf=1.5, cov="low"),
    "1.19": dict(K=12, cf=1.1, cov="low"),
    "1.20": dict(K=12, cf=1.5, cov="low"),
    "1.21": dict(K=13, cf=1.1, cov="low"),
    "1.22": dict(K=13, cf=1.5, cov="low"),
    "1.23": dict(K=14, cf=1.1, cov="low"),
    "1.24": dict(K=14, cf=1.5, cov="low"),
    "1.25": dict(K=15, cf=1.1, cov="low"),
    "1.26": dict(K=15, cf=1.5, cov="low"),

    # -------------------------
    # High demand variability
    # -------------------------
    "1.9":  dict(K=7,  cf=1.1, cov="high"),
    "1.10": dict(K=7,  cf=1.5, cov="high"),
    "1.11": dict(K=8,  cf=1.1, cov="high"),
    "1.12": dict(K=8,  cf=1.5, cov="high"),
    "1.13": dict(K=9,  cf=1.1, cov="high"),
    "1.14": dict(K=9,  cf=1.5, cov="high"),
    "1.15": dict(K=10, cf=1.1, cov="high"),
    "1.16": dict(K=10, cf=1.5, cov="high"),

    "1.27": dict(K=11, cf=1.1, cov="high"),
    "1.28": dict(K=11, cf=1.5, cov="high"),
    "1.29": dict(K=12, cf=1.1, cov="high"),
    "1.30": dict(K=12, cf=1.5, cov="high"),
    "1.31": dict(K=13, cf=1.1, cov="high"),
    "1.32": dict(K=13, cf=1.5, cov="high"),
    "1.33": dict(K=14, cf=1.1, cov="high"),
    "1.34": dict(K=14, cf=1.5, cov="high"),
    "1.35": dict(K=15, cf=1.1, cov="high"),
    "1.36": dict(K=15, cf=1.5, cov="high"),
}

# Case II, subset 1 (K=4): paper IDs 2.1..2.7 map to case_id 12..18
CASE_II_K4_REGISTRY = {
    "2.1": dict(case_id=12),
    "2.2": dict(case_id=13),
    "2.3": dict(case_id=14),
    "2.4": dict(case_id=15),
    "2.5": dict(case_id=16),
    "2.6": dict(case_id=17),
    "2.7": dict(case_id=18),
}

# Case II, subset 2 (K=10 patterns): paper IDs 2.8..2.14 map to demand_case_id 1..7
CASE_II_K10_REGISTRY = {
    "2.8":  dict(demand_case_id=1),
    "2.9":  dict(demand_case_id=2),
    "2.10": dict(demand_case_id=3),
    "2.11": dict(demand_case_id=4),
    "2.12": dict(demand_case_id=5),
    "2.13": dict(demand_case_id=6),
    "2.14": dict(demand_case_id=7),
}

# ============================================================
# 2) Instance model (aligned with DeepRL)
# ============================================================

class CLSPInstance:
    """
    Lightweight container for a CLSP instance, designed to match the DeepRL environment
    parameterization.

    This class stores the per-product parameters (demand mean, batch sizes, holding /
    backorder / setup costs, setup times, and demand variability proxy), plus the capacity
    factor used to derive the period capacity.

    :param K: Number of products.
    :type K: int
    :param capacity_factor: Capacity multiplier used to derive the per-period capacity.
    :type capacity_factor: float
    :param mu: Mean demand vector (one value per product).
    :type mu: array-like
    :param bs: Batch size vector (units produced per batch, one value per product).
    :type bs: array-like
    :param h: Holding cost vector (per-unit, per-period).
    :type h: array-like
    :param b: Backorder cost vector (per-unit, per-period).
    :type b: array-like
    :param k: Setup cost vector (per setup occurrence).
    :type k: array-like
    :param st: Setup time vector (capacity units consumed by a setup).
    :type st: array-like
    :param cov: Demand coefficient-of-variation proxy (used to select the demand range).
    :type cov: array-like
    """
    def __init__(self, K, capacity_factor, mu, bs, h, b, k, st, cov):
        self.K = int(K)
        self.capacity_factor = float(capacity_factor)
        self.mu = np.asarray(mu, dtype=np.float32)
        self.bs = np.asarray(bs, dtype=np.int32)
        self.h  = np.asarray(h, dtype=np.float32)
        self.b  = np.asarray(b, dtype=np.float32)
        self.k  = np.asarray(k, dtype=np.float32)
        self.st = np.asarray(st, dtype=np.int32)
        self.cov = np.asarray(cov, dtype=np.float32)

        assert self.mu.shape == (self.K,)
        assert self.bs.shape == (self.K,)
        assert self.h.shape  == (self.K,)
        assert self.b.shape  == (self.K,)
        assert self.k.shape  == (self.K,)
        assert self.st.shape == (self.K,)
        assert self.cov.shape == (self.K,)

    def capacity(self):
        """
        Compute the per-period capacity CAP, aligned with DeepRL's make_env_from_hyp() convention:

            CAP = ceil(capacity_factor * (sum(mu) / bs[0]))

        :return: Per-period capacity (in batches-equivalent units).
        :rtype: int
        """
        cap = math.ceil(self.capacity_factor * (float(self.mu.sum()) / float(self.bs[0])))
        return int(cap)

# ============================================================
# 3) Instance builders (parallel to the DeepRL HYP builders)
# ============================================================

def build_inst_set1(K, cov_level, cf):
    """
    Build a CLSPInstance corresponding to Set 1 in the comparison paper.

    Set 1 is homogeneous across products:
      - mu_i = 4
      - h_i = 1
      - b_i = 9
      - TBO_i = 10  -> defines setup costs k_i
      - st_i = 0
      - cov = 0.14 ("low") or 0.58 ("high")
      - bs_i constant across products

    Capacity is derived from capacity_factor (cf) via:
      CAP = ceil(cf * (sum(mu) / bs[0]))

    :param K: Number of products.
    :type K: int
    :param cov_level: Demand variability level ("low" or "high").
    :type cov_level: str
    :param cf: Capacity factor (1.1 or 1.5).
    :type cf: float
    :return: Fully specified CLSPInstance for Set 1.
    :rtype: CLSPInstance
    """
    assert cov_level in ("low", "high")
    assert cf in (1.1, 1.5)

    MU  = 4.0
    H   = 1.0
    B   = 9.0
    TBO = 10.0

    bs_val = (K // 2) * 2 # or 5 or 10 for scalability cases  
    COV = 0.14 if cov_level == "low" else 0.58

    mu  = np.full(K, MU, dtype=np.float32)
    bs  = np.full(K, bs_val, dtype=np.int32)
    h   = np.full(K, H, dtype=np.float32)
    b   = np.full(K, B, dtype=np.float32)
    cov = np.full(K, COV, dtype=np.float32)

    # k_i = h_i * TBO_i^2 * mu_i / 2
    TBO_vec = np.full(K, TBO, dtype=np.float32)
    k = H * (TBO_vec ** 2) * mu / 2.0

    st = np.zeros(K, dtype=np.int32)

    return CLSPInstance(K=K, capacity_factor=cf, mu=mu, bs=bs, h=h, b=b, k=k, st=st, cov=cov)


def build_inst_set2_case_K4(case_id, cf=1.1):
    """
    Build a CLSPInstance corresponding to Set 2 (K = 4, cases 12–18) in the comparison paper.

    Base structure:
      - K = 4
      - mu = [2, 2, 4, 4]
      - cov = 0.58
      - bs = 4, h = 1, st = 0
      - b and TBO depend on case_id
      - k_i computed from TBO via k_i = h_i * TBO_i^2 * mu_i / 2

    :param case_id: Case identifier in {12, 13, 14, 15, 16, 17, 18}.
    :type case_id: int
    :param cf: Capacity factor.
    :type cf: float
    :return: Fully specified CLSPInstance for the selected Set 2 case.
    :rtype: CLSPInstance
    """
    assert case_id in (12, 13, 14, 15, 16, 17, 18)

    K = 4
    MU_vec = np.array([2.0, 2.0, 4.0, 4.0], dtype=np.float32)
    H = 1.0
    COV = 0.58
    bs_val = 4

    mu  = MU_vec.copy()
    bs  = np.full(K, bs_val, dtype=np.int32)
    h   = np.full(K, H, dtype=np.float32)
    cov = np.full(K, COV, dtype=np.float32)
    st  = np.zeros(K, dtype=np.int32)

    if case_id == 12:
        TBO_vec = np.full(K, 5.0, dtype=np.float32)
        b_vec   = np.full(K, 9.0, dtype=np.float32)
    elif case_id == 13:
        TBO_vec = np.full(K, 10.0, dtype=np.float32)
        b_vec   = np.full(K, 9.0, dtype=np.float32)
    elif case_id == 14:
        TBO_vec = np.full(K, 5.0, dtype=np.float32)
        b_vec   = np.full(K, 49.0, dtype=np.float32)
    elif case_id == 15:
        TBO_vec = np.full(K, 10.0, dtype=np.float32)
        b_vec   = np.full(K, 49.0, dtype=np.float32)
    elif case_id == 16:
        TBO_vec = np.array([5.0, 5.0, 10.0, 10.0], dtype=np.float32)
        b_vec   = np.array([9.0, 9.0, 49.0, 49.0], dtype=np.float32)
    elif case_id == 17:
        TBO_vec = np.array([10.0, 10.0, 5.0, 5.0], dtype=np.float32)
        b_vec   = np.array([49.0, 49.0, 9.0, 9.0], dtype=np.float32)
    else:
        TBO_vec = np.array([5.0, 10.0, 10.0, 5.0], dtype=np.float32)
        b_vec   = np.array([9.0, 49.0, 9.0, 49.0], dtype=np.float32)

    b = b_vec
    k = H * (TBO_vec ** 2) * mu / 2.0

    return CLSPInstance(K=K, capacity_factor=cf, mu=mu, bs=bs, h=h, b=b, k=k, st=st, cov=cov)


def build_inst_set3_demand_case_K10(demand_case_id, cf=1.1, cov_level="high"):
    """
    Build a CLSPInstance for Set 3 custom demand scenarios (K = 10), aligned with the
    DeepRL environment's demand setup.
    
    Set 3 shared parameters:
      - K = 10
      - h = 1, b = 9, st = 0
      - fixed TBO = 10 -> k_i = h * TBO^2 * mu_i / 2
      - cov = 0.58 ("high") or 0.14 ("low")
      - bs = 4

    Demand cases (NOTE: case 4 and 7 are swapped as requested):
      1) Two-block:        [2]*5 + [8]*5
      2) Three-block:      [2]*4 + [4]*3 + [8]*3
      3) Mixed stepped:    [2]*3 + [3]*3 + [6,7,8,9]
      4) Pair +1 (corr):   mu=[2,2,4,4,6,6,8,8,10,10], corr="pair_plus_one"
      5) Pair same (corr): mu=[2,2,4,4,6,6,8,8,10,10], corr="pair_same"
      6) Group same (corr):mu=[2,2,2,2, 4,4,4, 8,8,8], corr="group_same"
      7) Increasing:       [2,3,4,5,6,7,8,9,10,11]
    """
    assert demand_case_id in (1, 2, 3, 4, 5, 6, 7)
    assert cov_level in ("low", "high")

    K = 10
    H = 1.0
    B = 9.0
    TBO = 10.0
    COV = 0.58 if cov_level == "high" else 0.14
    bs_val = 10

    # --- Default: no correlation ---
    corr_mode = None
    corr_pairs = None
    corr_groups = None

    # --- Demand cases (mu vectors) ---
    if demand_case_id == 1:
        mu = np.array([2]*5 + [8]*5, dtype=np.float32)

    elif demand_case_id == 2:
        mu = np.array([2]*4 + [4]*3 + [8]*3, dtype=np.float32)

    elif demand_case_id == 3:
        mu = np.array([2]*3 + [3]*3 + [6, 7, 8, 9], dtype=np.float32)

    elif demand_case_id == 4:
        # Case 4: pair_plus_one correlation
        mu = np.array([2, 2, 4, 4, 6, 6, 8, 8, 10, 10], dtype=np.float32)
        corr_mode = "pair_plus_one"
        corr_pairs = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    elif demand_case_id == 5:
        # Case 5: pair_same correlation
        mu = np.array([2, 2, 4, 4, 6, 6, 8, 8, 10, 10], dtype=np.float32)
        corr_mode = "pair_same"
        corr_pairs = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    elif demand_case_id == 6:
        # Case 6: group_same correlation
        mu = np.array([2]*4 + [4]*3 + [8]*3, dtype=np.float32)
        corr_mode = "group_same"
        corr_groups = [
            [0, 1, 2, 3],  # mu=2 group
            [4, 5, 6],     # mu=4 group
            [7, 8, 9],     # mu=8 group
        ]

    else:  # demand_case_id == 7
        # Case 7: strictly increasing
        mu = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=np.float32)

    # --- Shared vectors ---
    bs  = np.full(K, bs_val, dtype=np.int32)
    h   = np.full(K, H, dtype=np.float32)
    b   = np.full(K, B, dtype=np.float32)
    cov = np.full(K, COV, dtype=np.float32)
    st  = np.zeros(K, dtype=np.int32)

    # k_i from fixed TBO
    TBO_vec = np.full(K, TBO, dtype=np.float32)
    k = H * (TBO_vec ** 2) * mu / 2.0

    inst = CLSPInstance(K=K, capacity_factor=cf, mu=mu, bs=bs, h=h, b=b, k=k, st=st, cov=cov)

    # Attach correlation metadata (consumed by DemandGenerator)
    inst.demand_corr_mode = corr_mode
    inst.demand_corr_pairs = corr_pairs
    inst.demand_corr_groups = corr_groups

    return inst


def build_inst_from_instance_id(instance_id):
    instance_id = instance_id.strip()

    # Case I: homogeneous demand (Set 1)
    if instance_id in CASE_I_REGISTRY:
        p = CASE_I_REGISTRY[instance_id]
        inst = build_inst_set1(K=p["K"], cov_level=p["cov"], cf=p["cf"])
        exp_name = f"ID{instance_id}_K{p['K']}_cov{p['cov']}_cf{p['cf']}"
        return inst, exp_name

    # Case II subset 1: K=4 (Set 2)
    if instance_id in CASE_II_K4_REGISTRY:
        p = CASE_II_K4_REGISTRY[instance_id]
        inst = build_inst_set2_case_K4(case_id=p["case_id"], cf=1.1)
        exp_name = f"ID{instance_id}_case{p['case_id']}_cf1.1"
        return inst, exp_name

    # Case II subset 2: K=10 patterns (Set 3)
    if instance_id in CASE_II_K10_REGISTRY:
        p = CASE_II_K10_REGISTRY[instance_id]
        inst = build_inst_set3_demand_case_K10(
            demand_case_id=p["demand_case_id"],
            cf=1.1,
            cov_level="high",
        )
        exp_name = f"ID{instance_id}_dcase{p['demand_case_id']}_cf1.1"
        return inst, exp_name

    raise ValueError(f"Unknown INSTANCE_ID '{instance_id}' for S-CLSP.")


# ==========================================================================
# 4) Demand (identical to CLSPHierEnv._setup_demand_params + _sample_demand)
# ==========================================================================

class DemandGenerator:
    """
    Discrete demand generator aligned with the demand logic implemented in
    CLSPHierEnv._setup_demand_params() and CLSPHierEnv._sample_demand().

    It precomputes per-product integer intervals [a_i, b_i] defining a discrete
    uniform distribution and samples demands with rng.integers().

    :param inst: CLSP instance containing product parameters (mu, cov, etc.).
    :type inst: CLSPInstance
    :param rng: NumPy random generator used for reproducible demand sampling.
    :type rng: numpy.random.Generator
    """
    def __init__(self, inst: CLSPInstance, rng: np.random.Generator):
        self.inst = inst
        self.rng = rng
        self._setup_params()

    def _setup_params(self):
        """
        Precompute the per-product discrete uniform demand ranges [a_i, b_i] using the
        same convention as CLSPHierEnv._setup_demand_params().

        :return: None.
        :rtype: None
        """
        K = self.inst.K
        a = np.empty(K, dtype=np.int32)
        b = np.empty(K, dtype=np.int32)

        for i in range(K):
            cov_i = float(self.inst.cov[i])
            mu_i  = float(self.inst.mu[i])

            if abs(cov_i - 0.58) <= 1e-3:
                a[i] = 0
                b[i] = int(round(2.0 * mu_i))
            else:
                if abs(mu_i - 4.0) <= 1e-3:
                    a[i], b[i] = 3, 5
                else:
                    a[i] = b[i] = int(round(mu_i))

        self._a = a
        self._b = b

    def sample_period(self):
        """
        Sample one demand vector for the current period.
    
        Default behavior:
          - independent discrete-uniform demand per product: d_i ~ U{a_i, b_i}
    
        Optional correlated modes (must mirror CLSPHierEnv._sample_demand()):
          - "pair_plus_one": for each (i,j), sample base for i, set d[j] = base + 1 (clipped)
          - "pair_same":     for each (i,j), sample base for i, set d[j] = base (clipped)
          - "group_same":    for each group, sample one base from first member, set all to base (clipped)
        """
        d = self.rng.integers(low=self._a, high=self._b + 1, size=self.inst.K, dtype=np.int32)
    
        mode = getattr(self.inst, "demand_corr_mode", None)
    
        if mode == "pair_plus_one":
            pairs = getattr(self.inst, "demand_corr_pairs", None)
            if pairs is not None:
                for (i, j) in pairs:
                    base = self.rng.integers(self._a[i], self._b[i] + 1, dtype=np.int32)
                    d[i] = base
                    d[j] = base + 1
                    d[j] = np.clip(d[j], self._a[j], self._b[j])
    
        elif mode == "pair_same":
            pairs = getattr(self.inst, "demand_corr_pairs", None)
            if pairs is not None:
                for (i, j) in pairs:
                    base = self.rng.integers(self._a[i], self._b[i] + 1, dtype=np.int32)
                    d[i] = base
                    d[j] = base
                    d[i] = np.clip(d[i], self._a[i], self._b[i])
                    d[j] = np.clip(d[j], self._a[j], self._b[j])
    
        elif mode == "group_same":
            groups = getattr(self.inst, "demand_corr_groups", None)
            if groups is not None:
                for grp in groups:
                    grp = list(grp)
                    if len(grp) == 0:
                        continue
                    i0 = grp[0]
                    base = self.rng.integers(self._a[i0], self._b[i0] + 1, dtype=np.int32)
                    for j in grp:
                        d[j] = base
                        d[j] = np.clip(d[j], self._a[j], self._b[j])
    
        return d


# ============================================================
# 5) EOQ / Thresholds (aligned with the AMBS heuristic)
# ============================================================

def eoq(mu, k, h):
    """
    Compute the (continuous) Economic Order Quantity (EOQ) per product:

        Q_i = sqrt(2 * mu_i * k_i / h_i)

    :param mu: Mean demand rate(s) (scalar or array).
    :type mu: float or numpy.ndarray
    :param k: Setup/fixed cost(s) (scalar or array).
    :type k: float or numpy.ndarray
    :param h: Holding cost rate(s) (scalar or array).
    :type h: float or numpy.ndarray
    :return: EOQ quantities (broadcasted shape).
    :rtype: numpy.ndarray
    """
    return np.sqrt(2.0 * mu * k / h)


def ceoq(mu, k, h):
    """
    Compute the EOQ-based average cost proxy used by AMBS:

        C_i = k_i * mu_i / Q_i + h_i * Q_i / 2

    where Q_i is EOQ from eoq(mu, k, h).

    :param mu: Mean demand rate(s) (scalar or array).
    :type mu: float or numpy.ndarray
    :param k: Setup/fixed cost(s) (scalar or array).
    :type k: float or numpy.ndarray
    :param h: Holding cost rate(s) (scalar or array).
    :type h: float or numpy.ndarray
    :return: EOQ-based cost proxy values (broadcasted shape).
    :rtype: numpy.ndarray
    """
    Q = eoq(mu, k, h)
    return k * mu / Q + h * Q / 2.0


def make_thresholds(inst, xb, xh, zmax):
    """
    Build AMBS thresholds (Bmin, Hmax, Zmax) from scaling parameters:

      Bmin = xb * mean(C_i)
      Hmax = xh * sum(h_i * Q_i)
      Zmax = int(zmax)

    :param inst: CLSP instance containing mu, k, h.
    :type inst: CLSPInstance
    :param xb: Scaling factor for Bmin.
    :type xb: float
    :param xh: Scaling factor for Hmax.
    :type xh: float
    :param zmax: Maximum number of new setups per period.
    :type zmax: int or float
    :return: Tuple (Bmin, Hmax, Zmax).
    :rtype: tuple[float, float, int]
    """
    C = ceoq(inst.mu, inst.k, inst.h)
    Q = eoq(inst.mu, inst.k, inst.h)
    Bmin = xb * float(np.mean(C))
    Hmax = xh * float(np.sum(inst.h * Q))
    Zmax = int(zmax)
    return (Bmin, Hmax, Zmax)


class AMBSParams:
    """
    Container for AMBS threshold parameters.

    :param Bmin: Backorder trigger threshold.
    :type Bmin: float
    :param Hmax: Holding-cost threshold.
    :type Hmax: float
    :param Zmax: Maximum number of new setups allowed per period.
    :type Zmax: int
    """
    def __init__(self, Bmin, Hmax, Zmax):
        self.Bmin = float(Bmin)
        self.Hmax = float(Hmax)
        self.Zmax = int(Zmax)


class AMBSResult:
    """
    Result summary for an AMBS evaluation rollout.

    :param avg_cost: Mean total cost per period (holding + backorder + setup).
    :type avg_cost: float
    :param avg_inv_cost: Mean holding cost per period.
    :type avg_inv_cost: float
    :param avg_bo_cost: Mean backorder cost per period.
    :type avg_bo_cost: float
    :param avg_setup_cost: Mean setup cost per period.
    :type avg_setup_cost: float
    :param avg_setups: Mean number of setups per period.
    :type avg_setups: float
    :param avg_lot_size_units: Mean produced units per period.
    :type avg_lot_size_units: float
    :param step3_count: Number of periods in which Step 3 was executed.
    :type step3_count: int
    """
    def __init__(self, avg_cost, avg_inv_cost, avg_bo_cost, avg_setup_cost,
                 avg_setups, avg_lot_size_units, step3_count):
        self.avg_cost = float(avg_cost)
        self.avg_inv_cost = float(avg_inv_cost)
        self.avg_bo_cost = float(avg_bo_cost)
        self.avg_setup_cost = float(avg_setup_cost)
        self.avg_setups = float(avg_setups)
        self.avg_lot_size_units = float(avg_lot_size_units)
        self.step3_count = int(step3_count)

# ============================================================
# 6) AMBS Heuristic (adapted for strict comparability)
# ============================================================

class AMBSHeuristic:
    """
    AMBS heuristic simulator aligned with the DeepRL evaluation protocol.

    This class implements an AMBS-style production heuristic and is designed to be
    comparable with PPO_Hier.test() by matching:
      - per-run seed strategy (seed_base + r),
      - the discrete-uniform demand model,
      - the reset initialization (I = 0, random initial setup).

    :param inst: CLSPInstance with CLSP parameters.
    :type inst: CLSPInstance
    :param seed: Optional seed controlling initial setup and demand stream.
    :type seed: int or None
    """
    def __init__(self, inst: CLSPInstance, seed=None):
        self.inst = inst
        self.rng = np.random.default_rng(seed)
        self.dem = DemandGenerator(inst, self.rng)
        self.CAP = inst.capacity()
        self.Q_eoq = eoq(inst.mu, inst.k, inst.h)

    def _expected_backorder_cost(self, I):
        """
        Compute expected backorder cost per product under discrete-uniform demand.

        :param I: Inventory vector (positive = on-hand, negative = backlog).
        :type I: numpy.ndarray
        :return: Expected backorder costs per product, shape (K,).
        :rtype: numpy.ndarray
        """
        out = np.zeros(self.inst.K, dtype=np.float64)
        a = self.dem._a
        b = self.dem._b

        for i in range(self.inst.K):
            s  = float(I[i])
            ai = int(a[i])
            bi = int(b[i])

            if s >= bi:
                out[i] = 0.0
                continue

            denom = (bi - ai)
            if denom <= 0:
                d = ai
                ebo_units = max(0.0, d - s)
                out[i] = float(self.inst.b[i]) * ebo_units
                continue

            x0 = max(int(math.floor(s)) + 1, ai)
            if x0 > bi:
                out[i] = 0.0
                continue

            n = (bi - x0 + 1)
            sum_x = n * (x0 + bi) / 2.0
            ebo_units = (sum_x - n * s) / float(denom)
            out[i] = float(self.inst.b[i]) * float(ebo_units)

        return out

    def _il_eoq(self, I):
        """
        Compute the IL/EOQ metric used by AMBS.

        :param I: Inventory vector.
        :type I: numpy.ndarray
        :return: IL/EOQ values.
        :rtype: numpy.ndarray
        """
        return I / self.Q_eoq

    def _cap_remaining(self, q, produced, iSU):
        """
        Compute remaining capacity based on batch load and setup times.

        :param q: Batches per product for current period.
        :type q: numpy.ndarray
        :param produced: List of products with q_i > 0.
        :type produced: list[int]
        :param iSU: Initial setup item index for the period.
        :type iSU: int
        :return: Remaining capacity (in batches-equivalent units).
        :rtype: int
        """
        setup_load = sum(int(self.inst.st[i]) for i in produced if i != iSU)
        batch_load = int(np.sum(q))
        return int(self.CAP - setup_load - batch_load)

    def _initial_state_like_deeprl(self):
        """
        Initialize state as CLSPHierEnv.reset() does: I=0 and random setup item.

        :return: Tuple (I0, iSU).
        :rtype: tuple[numpy.ndarray, int]
        """
        I0 = np.zeros(self.inst.K, dtype=np.int32)
        iSU = int(self.rng.integers(0, self.inst.K))
        return I0, iSU

    def simulate(self, params: AMBSParams, horizon=1000, warmup=100,
                 seed_rollout=None, show_pbar=False):
        """
        Simulate the AMBS heuristic and report average per-period costs after warmup.

        :param params: AMBSParams (Bmin, Hmax, Zmax).
        :type params: AMBSParams
        :param horizon: Number of periods to simulate.
        :type horizon: int
        :param warmup: Number of initial periods excluded from averaging.
        :type warmup: int
        :param seed_rollout: Rollout seed (typically seed_base + r).
        :type seed_rollout: int or None
        :param show_pbar: If True, show tqdm progress bar.
        :type show_pbar: bool
        :return: AMBSResult with averaged cost components and diagnostics.
        :rtype: AMBSResult
        """
        if seed_rollout is not None:
            self.rng = np.random.default_rng(int(seed_rollout))
            self.dem = DemandGenerator(self.inst, self.rng)

        K = self.inst.K
        bs = self.inst.bs
        h  = self.inst.h
        bcost = self.inst.b
        kcost = self.inst.k
        st = self.inst.st

        I, iSU = self._initial_state_like_deeprl()

        total_cost = inv_cost = bo_cost = setup_cost = 0.0
        total_setups = total_units = 0
        step3 = np.zeros(horizon, dtype=np.int8)

        it = tqdm(range(horizon), desc="AMBS", leave=False) if show_pbar else range(horizon)

        for t in it:
            q = np.zeros(K, dtype=np.int32)
            produced = []
            last_prod = None

            CAP_rem = self._cap_remaining(q, produced, iSU)

            while CAP_rem >= 1:
                EBO = self._expected_backorder_cost(I)
                i_best = int(np.argmax(EBO))

                needs_setup = (i_best not in produced) and (i_best != iSU)
                new_setups_cnt = len([p for p in produced if p != iSU])

                # ===== STEP 1 =====
                if (EBO[i_best] > params.Bmin) and (new_setups_cnt <= params.Zmax):
                    if needs_setup and (CAP_rem > int(st[i_best]) + 1):
                        q[i_best] = 1
                        produced.append(i_best)
                        I[i_best] = I[i_best] + int(bs[i_best]) * 1
                        CAP_rem = self._cap_remaining(q, produced, iSU)
                        continue
                    elif not needs_setup:
                        q[i_best] += 1
                        I[i_best] = I[i_best] + int(bs[i_best]) * 1
                        CAP_rem = self._cap_remaining(q, produced, iSU)
                        continue
                    else:
                        break

                # ===== STEP 2 =====
                elif len(produced) == 0:
                    hold_cost = float(bs[iSU] * h[iSU] + np.sum(h * np.maximum(I, 0)))
                    if hold_cost <= params.Hmax:
                        q[iSU] += 1
                        I[iSU] = I[iSU] + int(bs[iSU]) * 1
                        CAP_rem = self._cap_remaining(q, produced, iSU)
                        continue
                    else:
                        break

                # ===== STEP 3 =====
                else:
                    step3[t] = 1
                    Ksetup = sorted(set(produced + [iSU]))
                    Kprod = list(dict.fromkeys(produced))

                    ILEOQ = self._il_eoq(I)
                    iFP = min(Kprod, key=lambda i: ILEOQ[i])
                    last_prod = int(iFP)

                    added = False
                    for i_add in [i for i in sorted(Ksetup, key=lambda x: ILEOQ[x]) if i != iFP]:
                        hold_cost = float(bs[i_add] * h[i_add] + np.sum(h * np.maximum(I, 0)))
                        if hold_cost <= params.Hmax and not added:
                            q[i_add] += 1
                            I[i_add] = I[i_add] + int(bs[i_add]) * 1
                            added = True

                    if not added:
                        break

                    CAP_rem = self._cap_remaining(q, produced, iSU)

            new_setups = sum(1 for i in produced if (i != iSU and q[i] > 0))
            setup_c = float(np.sum([kcost[i] for i in produced if (i != iSU and q[i] > 0)]))

            iSU_next = int(last_prod) if last_prod is not None else int(iSU)

            iSU = iSU_next
            I = I - self.dem.sample_period()

            c_inv  = float(np.sum(h * np.maximum(I, 0)))
            c_back = float(np.sum(bcost * np.maximum(-I, 0)))
            c_tot = c_inv + c_back + setup_c

            if t >= warmup:
                total_cost += c_tot
                inv_cost += c_inv
                bo_cost += c_back
                setup_cost += setup_c
                total_setups += int(new_setups)
                total_units += int(np.sum(bs * q))

        denom = max(1, int(horizon - warmup))
        return AMBSResult(
            avg_cost=total_cost / denom,
            avg_inv_cost=inv_cost / denom,
            avg_bo_cost=bo_cost / denom,
            avg_setup_cost=setup_cost / denom,
            avg_setups=total_setups / denom,
            avg_lot_size_units=total_units / denom,
            step3_count=int(np.sum(step3)),
        )

    def grid_search(self, xb_grid=None, xh_grid=None, z_grid=None,
                    trials=10, horizon=1000, warmup=100, base_seed=12345):
        """
        Grid-search AMBS thresholds and return the best parameters (by mean cost across trials).

        :param xb_grid: Candidate xb values for Bmin scaling.
        :type xb_grid: list[float] or None
        :param xh_grid: Candidate xh values for Hmax scaling.
        :type xh_grid: list[float] or None
        :param z_grid: Candidate integer values for Zmax.
        :type z_grid: list[int] or None
        :param trials: Rollouts per parameter configuration.
        :type trials: int
        :param horizon: Simulation horizon per rollout.
        :type horizon: int
        :param warmup: Warmup periods excluded from averaging.
        :type warmup: int
        :param base_seed: Base seed used to create rollout seeds.
        :type base_seed: int
        :return: Tuple (best_params, best_result) from an independent eval seed.
        :rtype: tuple[AMBSParams, AMBSResult]
        """
        K = self.inst.K
        if xb_grid is None:
            xb_grid = [i / 10.0 for i in range(0, 11)]
        if xh_grid is None:
            xh_grid = [i / 10.0 for i in range(5, 11)]
        if z_grid is None:
            z_grid = list(range(1, max(1, K)))

        combos = [(xb, xh, z) for xb in xb_grid for xh in xh_grid for z in z_grid]
        seeds = [int(base_seed + r) for r in range(int(trials))]

        best_par = None
        best_cost = None

        with tqdm(total=len(combos) * len(seeds), desc="Grid AMBS", leave=False) as pbar:
            for xb, xh, z in combos:
                Bmin, Hmax, Zmax = make_thresholds(self.inst, xb, xh, z)

                costs = []
                for s in seeds:
                    res = self.simulate(
                        AMBSParams(Bmin, Hmax, Zmax),
                        horizon=horizon,
                        warmup=warmup,
                        seed_rollout=s,
                        show_pbar=False,
                    )
                    costs.append(res.avg_cost)
                    pbar.update(1)

                avg_cost = float(np.mean(costs))
                if (best_cost is None) or (avg_cost < best_cost):
                    best_cost = avg_cost
                    best_par = AMBSParams(Bmin, Hmax, Zmax)

        final_eval = self.simulate(
            best_par,
            horizon=horizon,
            warmup=warmup,
            seed_rollout=int(base_seed + 999),
            show_pbar=False,
        )
        return best_par, final_eval

# ==================================================================
# 7) Instance construction according to paper INSTANCE_ID
# ==================================================================
inst, exp_name = build_inst_from_instance_id(INSTANCE_ID)

# ===================================================================
# 8) Execution: (i) threshold calibration; (ii) comparable evaluation
# ===================================================================

ambs = AMBSHeuristic(inst, seed=SEED)

best_params, best_result = ambs.grid_search(
    xb_grid=[i / 10.0 for i in range(0, 11)],
    xh_grid=[i / 10.0 for i in range(5, 11)],
    z_grid=list(range(1, inst.K)),
    trials=10,
    horizon=1000,
    warmup=100,
    base_seed=12345,
)

print("\n==============================================")
print(f"AMBS experiment: {exp_name}")
print("== Best thresholds found (grid) ==")
print(f"Bmin = {best_params.Bmin:.4f}, Hmax = {best_params.Hmax:.4f}, Zmax = {best_params.Zmax}")
print(f"Cost (grid evaluation) = {best_result.avg_cost:.3f}")
print("==============================================\n")

# ===== FINAL evaluation: identical to our DeepRL trainer.test() =====
N_RUNS = 10
HORIZON = 10_000
WARMUP = 1_000

SEED_BASE_TEST = 12345

run_costs = []
run_inv_costs = []
run_bo_costs = []
run_setup_costs = []
run_setups_rate = []
run_lot_units = []

print("== Individual AMBS results (same seeds as DeepRL test) ==")
for r in range(N_RUNS):
    seed_r = SEED_BASE_TEST + r
    res_r = ambs.simulate(
        best_params,
        horizon=HORIZON,
        warmup=WARMUP,
        seed_rollout=seed_r,
        show_pbar=False,
    )

    print(f"\n-- Run {r + 1}/{N_RUNS} (seed {seed_r}) --")
    print(f"Average total cost : {res_r.avg_cost:.3f}")
    print(f" - Inventory : {res_r.avg_inv_cost:.3f}")
    print(f" - Backorder : {res_r.avg_bo_cost:.3f}")
    print(f" - Setups : {res_r.avg_setup_cost:.3f}")
    print(f"Setups / period : {res_r.avg_setups:.3f}")
    print(f"Avg units / lot : {res_r.avg_lot_size_units:.3f}")
    print(f"Step 3 triggers : {res_r.step3_count}")

    run_costs.append(res_r.avg_cost)
    run_inv_costs.append(res_r.avg_inv_cost)
    run_bo_costs.append(res_r.avg_bo_cost)
    run_setup_costs.append(res_r.avg_setup_cost)
    run_setups_rate.append(res_r.avg_setups)
    run_lot_units.append(res_r.avg_lot_size_units)

print("\n== Average over 10 runs (horizon = 10k, warmup = 1k) ==")
print(f"Average total cost : {float(np.mean(run_costs)):.3f}")
print(f" - Inventory : {float(np.mean(run_inv_costs)):.3f}")
print(f" - Backorder : {float(np.mean(run_bo_costs)):.3f}")
print(f" - Setups : {float(np.mean(run_setup_costs)):.3f}")
print(f"Setups / period : {float(np.mean(run_setups_rate)):.3f}")
print(f"Avg units / lot : {float(np.mean(run_lot_units)):.3f}")
print("==============================================")



AMBS experiment: ID2.8_dcase1_cf1.1
== Best thresholds found (grid) ==
Bmin = 25.0000, Hmax = 300.0000, Zmax = 1
Cost (grid evaluation) = 581.908

== Individual AMBS results (same seeds as DeepRL test) ==

-- Run 1/10 (seed 12345) --
Average total cost : 580.949
 - Inventory : 245.543
 - Backorder : 18.506
 - Setups : 316.900
Setups / period : 1.022
Avg units / lot : 50.023
Step 3 triggers : 7193

-- Run 2/10 (seed 12346) --
Average total cost : 583.017
 - Inventory : 245.385
 - Backorder : 19.410
 - Setups : 318.222
Setups / period : 1.027
Avg units / lot : 50.082
Step 3 triggers : 7206

-- Run 3/10 (seed 12347) --
Average total cost : 582.294
 - Inventory : 245.387
 - Backorder : 19.507
 - Setups : 317.400
Setups / period : 1.028
Avg units / lot : 50.217
Step 3 triggers : 7148

-- Run 4/10 (seed 12348) --
Average total cost : 582.416
 - Inventory : 245.632
 - Backorder : 19.073
 - Setups : 317.711
Setups / period : 1.027
Avg units / lot : 50.046
Step 3 triggers : 7169

-- Run 5/10 (